# Task success breakdown

This notebook visualizes per-task success columns (`task_i_success`) from `results/eval_results.csv`.

Use it to quickly inspect task-level strengths/weaknesses across runs.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "visualization" else Path.cwd().resolve()
CSV_PATH = REPO_ROOT / "results" / "eval_results.csv"

df = pd.read_csv(CSV_PATH)

task_cols = [c for c in df.columns if re.fullmatch(r"task_\d+_success", c)]
if not task_cols:
    raise ValueError("No task_*_success columns found")

print(f"Loaded {len(df)} rows")
print(f"Found {len(task_cols)} task success columns")

df[["experiment_name", "seed"] + task_cols].head(3)

In [ ]:
# Heatmap: experiment x task mean success
if "experiment_name" not in df.columns:
    raise ValueError("experiment_name column missing")

mean_by_exp = df.groupby("experiment_name")[task_cols].mean()

plt.figure(figsize=(max(8, 0.55 * len(task_cols)), max(4, 0.4 * len(mean_by_exp))))
sns.heatmap(mean_by_exp, annot=True, fmt=".2f", cmap="viridis", cbar=True)
plt.title("Mean task success by experiment")
plt.xlabel("Task")
plt.ylabel("Experiment")
plt.tight_layout()
plt.show()

In [ ]:
# Mean per-task performance across all rows
task_means = df[task_cols].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(x=task_means.index, y=task_means.values, color="teal")
plt.xticks(rotation=50, ha="right")
plt.ylabel("Mean success")
plt.title("Overall mean success per task")
plt.tight_layout()
plt.show()

task_means.to_frame("mean_success")

In [ ]:
# Optional: compare only OPD-named runs vs all others (if names follow eval_opd_*)
if "experiment_name" in df.columns:
    tmp = df.copy()
    tmp["is_opd"] = tmp["experiment_name"].fillna("").str.contains("opd", case=False)
    opd_group = tmp.groupby("is_opd")[task_cols].mean()

    plt.figure(figsize=(max(8, 0.55 * len(task_cols)), 3.5))
    sns.heatmap(opd_group, annot=True, fmt=".2f", cmap="magma", cbar=True)
    plt.title("Task success: OPD-labeled runs vs others")
    plt.xlabel("Task")
    plt.ylabel("is_opd")
    plt.tight_layout()
    plt.show()

    opd_group